In [1]:
import polars as pl
import scanpy as sc
from lets_plot import *

import cellestial as cl

LetsPlot.setup_html()

import numpy as np
from lets_plot import *
from scipy import ndimage
from scipy.stats import gaussian_kde

data = sc.read("data/pbmc3k_pped.h5ad")

In [2]:
dim = cl.dimensional(
    data,
    dimensions="tsne",
    key="leiden",
    size=1,
    axis_type="arrow",
    alpha=0.6,
    tooltips=["leiden", "n_genes", "total_counts_hb", "MT-ND2"],
    legend_ondata=True,
    ondata_size=12,
    ondata_color="black",
    ondata_fontface="bold",
    ondata_family="sans",
    ondata_alpha=0.8,
)
dim += ggsize(800, 600)
dim

In [3]:
frame = cl.retrieve(dim)

In [4]:
def get_density_boundary(frame, x, y, group_by, padding=1.0, level=0.1, grid_size=100):
    boundaries = []
    groups = frame.get_column(group_by).unique().to_list()

    for g in groups:
        # 1. Isolate cluster points
        pts = frame.filter(pl.col(group_by) == g).select([x, y]).to_numpy()
        if len(pts) < 5:
            continue

        # 2. Kernel Density Estimation
        kde = gaussian_kde(pts.T)

        # 3. Create grid
        x_min, y_min = pts.min(axis=0) - padding
        x_max, y_max = pts.max(axis=0) + padding
        xi, yi = np.mgrid[x_min : x_max : grid_size * 1j, y_min : y_max : grid_size * 1j]

        # 4. Evaluate KDE
        zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)

        # 5. Extract Contour
        threshold = zi.max() * level

        # Fixed: Use get_paths() directly on the contour set
        cs = plt.contour(xi, yi, zi, levels=[threshold])

        paths = cs.get_paths()
        for i, path in enumerate(paths):
            v = path.vertices
            # We add a 'path_id' to ensure lets-plot doesn't connect separate islands
            boundaries.append(
                pl.DataFrame(
                    {
                        x: v[:, 0],
                        y: v[:, 1],
                        group_by: [g] * len(v),
                        "path_id": [f"{g}_{i}"] * len(v),
                    }
                )
            )

        plt.close()  # Keep the backend clean

    return pl.concat(boundaries)

In [5]:
density_frame = get_density_boundary(
    frame, x="X_TSNE1", y="X_TSNE2", group_by="leiden", padding=1.5, level=0.15, grid_size=100
)

NameError: name 'plt' is not defined

In [ ]:
density_frame

X_TSNE1,X_TSNE2,leiden,path_id
f64,f64,str,str
-28.877796,6.271942,"""0""","""0_0"""
-29.008067,7.173996,"""0""","""0_0"""
-29.102703,8.07605,"""0""","""0_0"""
-29.164016,8.9781,"""0""","""0_0"""
-29.192293,9.880154,"""0""","""0_0"""
…,…,…,…
-64.270283,-3.388935,"""12""","""12_0"""
-64.322876,-3.331567,"""12""","""12_0"""
-65.247902,-2.462517,"""12""","""12_0"""


In [ ]:
dim + geom_path(
    data=density_frame,
    mapping=aes(x="X_TSNE1", y="X_TSNE2", group="path_id"),
    color="black",
    linetype="dashed",
    size=1,
)

In [ ]:
from lets_plot import *


def get_density_boundary_pure(frame, x, y, group_by, padding=1.5, level=0.15, grid_size=100):
    boundaries = []
    groups = frame.get_column(group_by).unique().to_list()

    for g in groups:
        # 1. Isolate cluster points
        pts = frame.filter(pl.col(group_by) == g).select([x, y]).to_numpy()
        if len(pts) < 5:
            continue

        # 2. Kernel Density Estimation
        kde = gaussian_kde(pts.T)

        # 3. Create grid boundaries
        x_min, y_min = pts.min(axis=0) - padding
        x_max, y_max = pts.max(axis=0) + padding
        x_range = np.linspace(x_min, x_max, grid_size)
        y_range = np.linspace(y_min, y_max, grid_size)
        xi, yi = np.meshgrid(x_range, y_range)

        # 4. Evaluate KDE
        zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)

        # 5. Threshold to create a binary mask
        threshold = zi.max() * level
        mask = zi > threshold

        # 6. Extract the perimeter using SciPy (Binary Erosion)
        # This finds pixels that are on the edge of the density "blob"
        erosion = ndimage.binary_erosion(mask)
        perimeter = mask ^ erosion  # XOR operation to get only the edge

        coords = np.argwhere(perimeter)
        if len(coords) == 0:
            continue

        # 7. Convert pixel indices back to UMAP space
        # We sort them by angle from the center to ensure a smooth path
        center = coords.mean(axis=0)
        angles = np.arctan2(coords[:, 0] - center[0], coords[:, 1] - center[1])
        sorted_indices = np.argsort(angles)
        sorted_coords = coords[sorted_indices]

        # Add the first point at the end to close the loop
        sorted_coords = np.vstack([sorted_coords, sorted_coords[0]])

        actual_x = np.interp(sorted_coords[:, 1], np.arange(grid_size), x_range)
        actual_y = np.interp(sorted_coords[:, 0], np.arange(grid_size), y_range)

        boundaries.append(
            pl.DataFrame(
                {
                    x: actual_x,
                    y: actual_y,
                    group_by: [g] * len(actual_x),
                    "path_id": [f"{g}_0"] * len(actual_x),
                }
            )
        )

    return pl.concat(boundaries)


# --- Usage with your 'dim' object ---
density_frame = get_density_boundary_pure(
    frame, x="X_TSNE1", y="X_TSNE2", group_by="leiden", padding=1.5, level=0.1, grid_size=120
)

dim + geom_path(
    data=density_frame,
    mapping=aes(x="X_TSNE1", y="X_TSNE2", group="path_id"),
    color="black",
    linetype="dashed",
    size=1,
)